# [16.1] Exact Shapley on Ground-Truth Games

This notebook starts the Shapley baseline track from a place where the answer is known. You will enumerate complete coalition tables, compute exact Shapley values, check efficiency and permutation parity, and then read the committed CUDA report for a trained neural coalition game.

<img src="../../instructions/assets/exact_shapley_validation_loop.svg" width="760">

The point is not to admire a plausible attribution vector. The point is to build a reference that later approximate SHAP, TokenSHAP, interaction, and patching sections can fail against.

<details><summary>Help - why start with a complete finite game?</summary>

If every coalition is known, then the attribution target is not estimated from a sampling scheme or a perturbation interface. It is exact. That makes mistakes in weighting, missing coalitions, and interaction handling visible before you scale the method.

</details>


## Setup

Run the setup cell once. The public tests are intentionally small and exact; they are there to catch conceptual bugs, not to benchmark speed.

<details><summary>Expected output</summary>

No printed output. Imports should succeed and the dataclasses should be defined.

</details>


In [ ]:
from collections.abc import Callable, Mapping
from dataclasses import dataclass
import itertools
import json
import math
import sys
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part1_exact_shapley_ground_truth_games"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_exact_shapley_ground_truth_games.tests as tests

Coalition = frozenset[int]


@dataclass(frozen=True)
class ShapleyEfficiencyReport:
    shapley_sum: float
    total_value_delta: float
    efficiency_error: float
    satisfies_efficiency: bool


@dataclass(frozen=True)
class PermutationParityReport:
    max_abs_error: float
    matches_exact: bool


@dataclass(frozen=True)
class InteractionGapReport:
    shapley_total: float
    leave_one_out_total: float
    overcount: float
    detects_interaction_overcount: bool


## Exercise 1 - enumerate complete coalition tables

Write the utilities that every later exact-Shapley check depends on. `all_coalitions(3)` should return eight `frozenset` coalitions, including the empty set and the full coalition. `normalize_coalition_values` should reject incomplete tables instead of treating missing values as zero.

<details><summary>Help - what counts as complete?</summary>

For `n` players, a complete table has exactly `2**n` entries. The empty coalition, every singleton, every interaction subset, and the full coalition all need explicit values.

</details>

<details><summary>Common bugs</summary>

- Returning mutable `set` keys, which cannot be dictionary keys.
- Omitting the empty coalition.
- Accepting incomplete tables and silently changing the game.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_all_coalitions_enumerates_the_power_set` passed!
```

</details>

<details><summary>Solution</summary>

Use `itertools.combinations` for every subset size and convert every key to a `frozenset` before comparing with the expected power set.

</details>


In [ ]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    raise NotImplementedError()


def normalize_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> dict[Coalition, float]:
    raise NotImplementedError()


def coalition_values_from_function(
    num_players: int,
    value_fn: Callable[[Coalition], float],
) -> dict[Coalition, float]:
    raise NotImplementedError()


tests.test_all_coalitions_enumerates_the_power_set(all_coalitions)


## Exercise 2 - compute exact Shapley values

Implement the weighted marginal-effect formula. In an additive game, exact Shapley values must recover the original feature weights, including negative weights.

<details><summary>Help - where do the factorial weights come from?</summary>

A coalition of size `s` appears before player `i` in `s! * (n-s-1)!` permutations out of `n!` total permutations. The closed-form Shapley formula writes that average directly.

</details>

<details><summary>Common bugs</summary>

- Including coalitions that already contain the player.
- Dividing by `2**num_players` instead of `num_players!`.
- Using `float32` and hiding exact parity failures.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_additive_game_and_exact_shapley_recover_weights` passed!
All tests in `test_exact_shapley_requires_a_complete_coalition_table` passed!
```

</details>

<details><summary>Solution</summary>

Loop over players, then over coalitions that exclude that player. Add the weighted difference between `v(S union {i})` and `v(S)`.

</details>


In [ ]:
def exact_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    raise NotImplementedError()


def additive_game(weights: t.Tensor) -> dict[Coalition, float]:
    raise NotImplementedError()


tests.test_additive_game_and_exact_shapley_recover_weights(
    additive_game,
    exact_shapley_values,
)
tests.test_exact_shapley_requires_a_complete_coalition_table(exact_shapley_values)


## Exercise 3 - check efficiency on an AND game

In a three-player conjunction game, only the full coalition has value one. Symmetry says the credit should split equally, and efficiency says total credit should equal `v(full) - v(empty)`.

<details><summary>Help - why is efficiency a hard sanity check?</summary>

Efficiency is a conservation law. If exact Shapley values do not sum to the full-minus-empty value on a complete table, the implementation is not exact Shapley.

</details>

<details><summary>Common bugs</summary>

- Returning one when any player is present rather than only for the full coalition.
- Comparing against `v(full)` when `v(empty)` is nonzero.
- Using exact equality where a tolerance belongs.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_conjunction_game_splits_symmetric_credit_and_checks_efficiency` passed!
```

</details>

<details><summary>Solution</summary>

Create the AND game with value one only for the full coalition, compute Shapley, and compare the sum to `v(full) - v(empty)`.

</details>


In [ ]:
def conjunction_game(num_players: int) -> dict[Coalition, float]:
    raise NotImplementedError()


def shapley_efficiency_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    tolerance: float = 1e-9,
) -> ShapleyEfficiencyReport:
    raise NotImplementedError()


tests.test_conjunction_game_splits_symmetric_credit_and_checks_efficiency(
    conjunction_game,
    exact_shapley_values,
    shapley_efficiency_report,
)


## Exercise 4 - compare with permutation averaging

Compute Shapley values again by averaging marginal contributions over every player ordering. This is an independent exact check on the closed-form weights.

<details><summary>Help - why compare two exact methods?</summary>

The formula and permutation view are mathematically equivalent, but they fail for different coding reasons. Matching them catches indexing, ordering, and weighting mistakes.

</details>

<details><summary>Common bugs</summary>

- Updating the coalition before measuring the marginal contribution.
- Dividing by the number of coalitions instead of permutations.
- Sampling permutations in this exact reference exercise.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_permutation_parity_report_matches_exact_formula` passed!
```

</details>

<details><summary>Solution</summary>

Average the marginal contribution of each player over all `num_players!` permutations, then compare against the closed-form result.

</details>


In [ ]:
def permutation_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    raise NotImplementedError()


def permutation_parity_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    tolerance: float = 1e-9,
) -> PermutationParityReport:
    raise NotImplementedError()


tests.test_permutation_parity_report_matches_exact_formula(
    conjunction_game,
    permutation_parity_report,
)


## Exercise 5 - expose leave-one-out overcounting

Leave-one-out scores can overcount when value comes from an interaction. In a two-player AND game, removing either player destroys all value, so leave-one-out assigns one point to each player even though total game value is one.

<details><summary>Help - what failure is leave-one-out showing?</summary>

Leave-one-out asks how much value disappears when one feature is removed from the full coalition. Shapley asks how much credit a player receives averaged over contexts. These differ sharply on interactions.

</details>

<details><summary>Common bugs</summary>

- Computing leave-one-out from the empty coalition.
- Treating the overcount as a Shapley failure rather than a baseline failure.
- Forgetting that other interactions can make perturbation scores undercount instead.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_interaction_gap_report_catches_leave_one_out_overcount` passed!
```

</details>

<details><summary>Solution</summary>

Measure the drop from the full coalition when each player is removed, then compare total leave-one-out credit to total Shapley credit.

</details>


In [ ]:
def leave_one_out_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    raise NotImplementedError()


def interaction_gap_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    min_overcount: float = 0.5,
) -> InteractionGapReport:
    raise NotImplementedError()


tests.test_interaction_gap_report_catches_leave_one_out_overcount(
    conjunction_game,
    interaction_gap_report,
)


## Exercise 6 - assemble the notebook contract and read the report

The smoke contract is fast, deterministic, and exact. The committed CUDA report is slower and checks the trained-model path: fit a small MLP on the complete binary game, recover analytic Shapley from real model ablations, and reject a shuffled-label model.

<details><summary>Help - why keep the CUDA path separate?</summary>

The notebook contract isolates the math you are implementing. The CUDA report asks whether the same exact-Shapley logic survives a trained model whose coalition table is recovered by ablation.

</details>

<details><summary>Common bugs</summary>

- Returning tensors directly instead of JSON-serializable values.
- Treating the toy contract as the GPU evidence.
- Claiming OOD generalization from a complete finite-domain table.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_additive_smoke_test` passed!
All tests in `test_conjunction_smoke_test` passed!
All tests in `test_permutation_parity_smoke_test` passed!
All tests in `test_interaction_failure_smoke_test` passed!
All tests in `test_notebook_contract` passed!
All tests in `test_committed_gpu_report_records_exact_shapley_preflight` passed!
```

</details>

<details><summary>Solution</summary>

The report reader should assert the committed evidence fields directly. It should not rerun CUDA from inside the learner notebook.

</details>


In [ ]:
def additive_smoke_test() -> dict:
    raise NotImplementedError()


def conjunction_smoke_test() -> dict:
    raise NotImplementedError()


def permutation_parity_smoke_test() -> dict:
    raise NotImplementedError()


def interaction_failure_smoke_test() -> dict:
    raise NotImplementedError()


def run_smoke_test(cpu: bool = True) -> dict:
    raise NotImplementedError()


def load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    gpu = report["metrics"]["gpu_test"]
    assert report["accepted"] and report["tests_passed"], "The committed report should be accepted."
    assert gpu["cuda_available"] and gpu["preflight_passed"], "The report should record a real CUDA run."
    assert gpu["coalition_count"] == 16, "The report should cover the complete binary feature table."
    assert gpu["complete_finite_domain_evaluated"] is True, "The finite table should be completely evaluated."
    assert gpu["ood_generalization_claimed"] is False, "Do not claim OOD generalization here."
    assert gpu["neural_shapley_max_abs_error"] <= 1e-4, "Model Shapley should recover analytic Shapley."
    assert gpu["shuffled_control_rejected"], "The shuffled-label control should fail."
    return report


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    _ = max_vram_gb
    return load_committed_gpu_report()["metrics"]["gpu_test"]


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


tests.test_additive_smoke_test(additive_smoke_test)
tests.test_conjunction_smoke_test(conjunction_smoke_test)
tests.test_permutation_parity_smoke_test(permutation_parity_smoke_test)
tests.test_interaction_failure_smoke_test(interaction_failure_smoke_test)
tests.test_notebook_contract(run_smoke_test)
tests.test_committed_gpu_report_records_exact_shapley_preflight()


## Signature Result

<img src="../../instructions/assets/exact_shapley_signature_result.svg" width="760">

| Check | Expected | Observed |
| --- | ---: | ---: |
| Complete coalition table | 16 | 16 |
| Neural-game fit MSE | <= 1e-8 | 1.102e-12 |
| Model Shapley max error | <= 1e-4 | 9.38e-7 |
| Efficiency error | <= 1e-8 | 0.0 |
| Shuffled-control error | >= 1.0 | 4.6333 |
| Shuffled-control cosine | <= 0.25 | -0.7646 |
| Peak VRAM | <= 1.0 GB | 0.063 GB |

<details><summary>Help - how should this result be interpreted?</summary>

The result says exact Shapley works on a fully enumerated neural coalition game and that a shuffled-label trained model fails the true attribution vector. It does not say sampled SHAP, TokenSHAP, or large-model explanations are trustworthy.

</details>


## Limitations

- This is a GT-0 complete-table model organism, not a broad large-model attribution result.
- Complete finite-domain evaluation is not OOD generalization.
- Exact Shapley is exponential in the number of players, so later sections introduce approximation and grouping controls.
- The shuffled-label control rejects one important failure mode, not every possible attribution artifact.

## Further Research

- Compare sampled KernelSHAP convergence against this exact reference.
- Replace the binary game with a small sparse feature circuit and compare to activation patching.
- Use the leave-one-out counterexample when reviewing papers that report perturbation-only feature importance.
